# 06 — Translation Preprocessing

This notebook prepares open-ended survey responses for NLP by selecting the configured text columns, filtering complete responses, detecting Khmer/English/mixed language usage, translating Khmer or mixed responses with TranslateKH, and saving an NLP-ready dataset.

## Configuration-first design

Change-prone values are stored in `ml/config/config.json` under `translation_preprocessing`, including input/output paths, selected columns, text columns, language labels, regex patterns, translation languages, TranslateKH timeout/API settings, output suffixes, and CSV options.

Secrets remain in `.env` or shell environment variables, not in `config.json`.

In [1]:
from pathlib import Path
import sys

# Make imports work from either project/ml or project/ml/notebook.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    src_dir = candidate / "src"
    config_file = candidate / "config" / "config.json"
    if src_dir.exists() and config_file.exists():
        ML_ROOT = candidate.resolve()
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break
else:
    raise FileNotFoundError("Could not find the ml project root containing config/config.json and src/.")

import re
import pandas as pd

from blended_learning.config.settings import settings
from blended_learning.utils.notebook import resolve_project_path
from blended_learning.utils.io import require_columns
from blended_learning.utils.decorator import execution_time
from blended_learning.nlp.translate_kh_service import TranslateKHService

cfg = settings.translation_preprocessing
io_cfg = cfg["io"]
column_cfg = cfg["columns"]
filter_cfg = cfg["filtering"]
detection_cfg = cfg["language_detection"]
translation_cfg = cfg["translation"]
display_cfg = cfg["display"]

INPUT_PATH = settings.path[io_cfg["input_path_key"]]
OUTPUT_PATH = resolve_project_path(io_cfg["output_path"], settings.root)
SELECTED_COLUMNS = column_cfg["selected_columns"]
TEXT_COLUMNS = column_cfg["text_columns"]
LANG_SUFFIX = column_cfg["language_suffix"]
TRANSLATED_SUFFIX = column_cfg["translated_suffix"]

print(f"ML root: {ML_ROOT}")
print(f"Input:  {INPUT_PATH}")
print(f"Output: {OUTPUT_PATH}")


ML root: C:\Users\Tepy\Documents\tepy\Final Internship Docs\Blended-Learning\ml
Input:  C:\Users\Tepy\Documents\tepy\Final Internship Docs\Blended-Learning\ml\data\processed\cluster_data.csv
Output: C:\Users\Tepy\Documents\tepy\Final Internship Docs\Blended-Learning\ml\data\processed\nlp_translated_responses.csv


In [2]:
df = pd.read_csv(INPUT_PATH, **io_cfg.get("read_csv_options", {}))

require_columns(df, SELECTED_COLUMNS, context="translation preprocessing input")

df_selected = df[SELECTED_COLUMNS].copy()

print(f"NLP source dataframe shape: {df_selected.shape}")
display(df_selected.head(n=display_cfg["preview_rows"]))

NLP source dataframe shape: (592, 5)


,student_id,open_strengths,open_challenges_suggestions,student_segment,student_segment_label
0,e20210528,NaN,NaN,1,Moderately Engaged (Passive) Learners
1,e20210686,Nothing,Nothing,1,Moderately Engaged (Passive) Learners
2,e20241146,"Very good, excellent","No big challenge, i’m the best",2,Highly Engaged (Active) Learners
3,e20240609,Will try hard,Lack of self-discipline,2,Highly Engaged (Active) Learners
4,e20240542,getting more experience,Discipline on daily studying,1,Moderately Engaged (Passive) Learners


In [3]:
def build_complete_text_mask(data: pd.DataFrame, text_columns: list[str]) -> pd.Series:
    """Return rows with non-empty values in every configured text column."""
    missing_tokens = {str(token).strip().lower() for token in filter_cfg.get("missing_tokens", [])}

    masks = []
    for col in text_columns:
        normalised = data[col].astype(str).str.strip().str.lower()
        masks.append(data[col].notna() & ~normalised.isin(missing_tokens))

    if not masks:
        raise ValueError("No text columns configured for translation preprocessing.")

    mask = masks[0]
    for item in masks[1:]:
        mask = mask & item
    return mask


print(f"Before filtering: {len(df_selected)}")

if filter_cfg.get("require_complete_text", True):
    complete_text_mask = build_complete_text_mask(df_selected, TEXT_COLUMNS)
else:
    complete_text_mask = pd.Series(True, index=df_selected.index)

df_nlp = df_selected.loc[complete_text_mask].copy()

print(f"After filtering: {len(df_nlp)}")

missing_preview = df_selected.loc[~complete_text_mask].head(display_cfg["missing_preview_rows"])
display(missing_preview)

Before filtering: 592
After filtering: 570


,student_id,open_strengths,open_challenges_suggestions,student_segment,student_segment_label
0,e20210528,NaN,NaN,1,Moderately Engaged (Passive) Learners
9,e20240956,NaN,NaN,1,Moderately Engaged (Passive) Learners
24,e20240618,NaN,NaN,1,Moderately Engaged (Passive) Learners
25,e20240594,schedule flexibility,NaN,2,Highly Engaged (Active) Learners
29,e20220370,NaN,NaN,1,Moderately Engaged (Passive) Learners


In [4]:
KHMER_REGEX = re.compile(detection_cfg["regex"]["khmer"])
EN_REGEX = re.compile(detection_cfg["regex"]["english"])
LANG_LABELS = detection_cfg["labels"]
LANG_SUMMARY_ORDER = detection_cfg["summary_order"]


def detect_lang_mixed(text: object) -> str:
    """Detect whether a response is Khmer, English, mixed, empty, or other."""
    if pd.isna(text):
        return LANG_LABELS["empty"]

    text = str(text).strip()
    if text == "":
        return LANG_LABELS["empty"]

    has_khmer = bool(KHMER_REGEX.search(text))
    has_english = bool(EN_REGEX.search(text))

    if has_khmer and has_english:
        return LANG_LABELS["mixed"]
    if has_khmer:
        return LANG_LABELS["khmer"]
    if has_english:
        return LANG_LABELS["english"]
    return LANG_LABELS["other"]


for col in TEXT_COLUMNS:
    df_nlp[f"{col}{LANG_SUFFIX}"] = df_nlp[col].apply(detect_lang_mixed)


def summarize_language_distribution(data: pd.DataFrame, text_columns: list[str]) -> pd.DataFrame:
    all_langs = []
    separator = "=" * display_cfg.get("summary_separator_width", 60)

    for col in text_columns:
        lang_col = f"{col}{LANG_SUFFIX}"
        langs = data[lang_col]
        all_langs.append(langs)

        counts = langs.value_counts().reindex(LANG_SUMMARY_ORDER, fill_value=0)
        perc = (counts / counts.sum() * 100).round(1) if counts.sum() else counts
        summary = pd.DataFrame({"count": counts, "percent (%)": perc})

        translation_needed = counts.reindex(
            translation_cfg["languages_requiring_translation"],
            fill_value=0,
        ).sum()
        translation_pct = round((translation_needed / counts.sum()) * 100, 1) if counts.sum() else 0

        print(f"\n{separator}")
        print(f"SUMMARY FOR: {col}")
        print(separator)
        print(summary)
        print(f"\nTotal needing translation: {translation_needed} ({translation_pct}%)")

    combined_langs = pd.concat(all_langs, axis=0)
    agg_counts = combined_langs.value_counts().reindex(LANG_SUMMARY_ORDER, fill_value=0)
    agg_perc = (agg_counts / agg_counts.sum() * 100).round(1) if agg_counts.sum() else agg_counts
    agg_summary = pd.DataFrame({"count": agg_counts, "percent (%)": agg_perc})

    agg_translation_needed = agg_counts.reindex(
        translation_cfg["languages_requiring_translation"],
        fill_value=0,
    ).sum()
    agg_translation_pct = round((agg_translation_needed / agg_counts.sum()) * 100, 1) if agg_counts.sum() else 0

    print(f"\n{separator}")
    print("OVERALL LANGUAGE SUMMARY")
    print(separator)
    print(agg_summary)
    print(f"\nOverall needing translation: {agg_translation_needed} ({agg_translation_pct}%)")

    return agg_summary


language_summary = summarize_language_distribution(df_nlp, TEXT_COLUMNS)


SUMMARY FOR: open_strengths
                     count  percent (%)
open_strengths_lang                    
kh                      24          4.2
mixed                    2          0.4
en                     538         94.4
other                    6          1.1

Total needing translation: 26 (4.6%)

SUMMARY FOR: open_challenges_suggestions
                                  count  percent (%)
open_challenges_suggestions_lang                    
kh                                   23          4.0
mixed                                 5          0.9
en                                  536         94.0
other                                 6          1.1

Total needing translation: 28 (4.9%)

OVERALL LANGUAGE SUMMARY
       count  percent (%)
kh        47          4.1
mixed      7          0.6
en      1074         94.2
other     12          1.1

Overall needing translation: 54 (4.7%)


## Translate Khmer and mixed responses

Only configured language labels are translated. English and other responses are copied unchanged.

In [5]:
translator = TranslateKHService()
translation_cache: dict[str, str] = {}


def translate_if_needed(text: object, lang: str) -> str:
    """Translate configured languages; otherwise keep the original text."""
    if pd.isna(text):
        return ""

    text = str(text).strip()
    if text == "":
        return ""

    if lang not in translation_cfg["languages_requiring_translation"]:
        return text

    if translation_cfg.get("cache_enabled", True) and text in translation_cache:
        return translation_cache[text]

    translated = translator.translate_one(
        text=text,
        src_lang=translation_cfg["src_lang"],
        tgt_lang=translation_cfg["tgt_lang"],
    )

    if translation_cfg.get("fallback_to_original_on_empty_translation", False) and not translated:
        translated = text

    if translation_cfg.get("cache_enabled", True):
        translation_cache[text] = translated

    return translated

In [6]:
@execution_time
def translate_column(data: pd.DataFrame, col: str) -> pd.DataFrame:
    translated_col = f"{col}{TRANSLATED_SUFFIX}"
    lang_col = f"{col}{LANG_SUFFIX}"

    data[translated_col] = data.apply(
        lambda row: translate_if_needed(row[col], row[lang_col]),
        axis=1,
    )

    print(f"Finished translating: {col}")
    return data


for col in TEXT_COLUMNS:
    df_nlp = translate_column(df_nlp, col)

TranslateKH API HTTP error: 401 Client Error: Unauthorized for url: https://www.translate.kh/api
Response text: {"error":"Unauthorized"}
TranslateKH API HTTP error: 401 Client Error: Unauthorized for url: https://www.translate.kh/api
Response text: {"error":"Unauthorized"}
TranslateKH API HTTP error: 401 Client Error: Unauthorized for url: https://www.translate.kh/api
Response text: {"error":"Unauthorized"}
TranslateKH API HTTP error: 401 Client Error: Unauthorized for url: https://www.translate.kh/api
Response text: {"error":"Unauthorized"}
TranslateKH API HTTP error: 401 Client Error: Unauthorized for url: https://www.translate.kh/api
Response text: {"error":"Unauthorized"}
TranslateKH API HTTP error: 401 Client Error: Unauthorized for url: https://www.translate.kh/api
Response text: {"error":"Unauthorized"}
TranslateKH API HTTP error: 401 Client Error: Unauthorized for url: https://www.translate.kh/api
Response text: {"error":"Unauthorized"}
TranslateKH API HTTP error: 401 Client Er

In [7]:
translation_preview_columns = []
for col in TEXT_COLUMNS:
    translation_preview_columns.extend([
        col,
        f"{col}{LANG_SUFFIX}",
        f"{col}{TRANSLATED_SUFFIX}",
    ])

display(df_nlp[translation_preview_columns].head(display_cfg["translation_preview_rows"]))

,open_strengths,open_strengths_lang,open_strengths_en,open_challenges_suggestions,open_challenges_suggestions_lang,open_challenges_suggestions_en
1,Nothing,en,Nothing,Nothing,en,Nothing
2,"Very good, excellent",en,"Very good, excellent","No big challenge, i’m the best",en,"No big challenge, i’m the best"
3,Will try hard,en,Will try hard,Lack of self-discipline,en,Lack of self-discipline
4,getting more experience,en,getting more experience,Discipline on daily studying,en,Discipline on daily studying
5,Student could retrieve the lesson once they wa...,en,Student could retrieve the lesson once they wa...,Add Ai assistance,en,Add Ai assistance
6,Innovate yourself with the technology world,en,Innovate yourself with the technology world,Lack of resources and misinformation,en,Lack of resources and misinformation
7,Blended learning has many positive aspects. It...,en,Blended learning has many positive aspects. It...,The biggest challenges of blended learning are...,en,The biggest challenges of blended learning are...
8,Give a chance for students who prefer both onl...,en,Give a chance for students who prefer both onl...,"Internet issues, technical problems, lack of i...",en,"Internet issues, technical problems, lack of i..."
10,Give me time to do my stuff,en,Give me time to do my stuff,Probably the internet connection,en,Probably the internet connection
11,Agree,en,Agree,Agree,en,Agree


In [8]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_nlp.to_csv(OUTPUT_PATH, **io_cfg.get("write_csv_options", {}))

print(f"Saved translated NLP dataset to: {OUTPUT_PATH}")
print(f"NLP-ready dataset shape: {df_nlp.shape}")
display(df_nlp.head(display_cfg["preview_rows"]))

Saved translated NLP dataset to: C:\Users\Tepy\Documents\tepy\Final Internship Docs\Blended-Learning\ml\data\processed\nlp_translated_responses.csv
NLP-ready dataset shape: (570, 9)


,student_id,open_strengths,open_challenges_suggestions,student_segment,student_segment_label,open_strengths_lang,open_challenges_suggestions_lang,open_strengths_en,open_challenges_suggestions_en
1,e20210686,Nothing,Nothing,1,Moderately Engaged (Passive) Learners,en,en,Nothing,Nothing
2,e20241146,"Very good, excellent","No big challenge, i’m the best",2,Highly Engaged (Active) Learners,en,en,"Very good, excellent","No big challenge, i’m the best"
3,e20240609,Will try hard,Lack of self-discipline,2,Highly Engaged (Active) Learners,en,en,Will try hard,Lack of self-discipline
4,e20240542,getting more experience,Discipline on daily studying,1,Moderately Engaged (Passive) Learners,en,en,getting more experience,Discipline on daily studying
5,e20220287,Student could retrieve the lesson once they wa...,Add Ai assistance,1,Moderately Engaged (Passive) Learners,en,en,Student could retrieve the lesson once they wa...,Add Ai assistance
